In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
if not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)],
        check=True,
    )
    print("Cloned branch 'running': ", CLONE_DIR)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)
    print("Pulled latest: ", CLONE_DIR)

CANDIDATE_SRC = [
    str(CLONE_DIR / "src"),
    "/kaggle/input/ariel-ml-src/src",
    "src", "../src",
]
for _p in CANDIDATE_SRC:
    if Path(_p).exists():
        sys.path.insert(0, _p)
        print("Using src from:", _p)
        break
else:
    print("WARNING: src not found.")

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR = OUTPUT_DIR / "weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT exists:", DATA_ROOT.exists())


## 1. Configs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import ModelConfig
from dataset_builder import align_features_and_targets
from benchmark import benchmark_models
from training import cross_validate_model, train_model, search_n_components, build_gll_weighted_ensemble

LIMIT = None
TIME_BINS = 128

N_COMPONENTS = 24
N_SPLITS = 3
SIGMA_CAL_FRACTION = 0.2
USE_GPU = False
RANDOM_STATE = 42
MAX_PLANETS_BENCH = None

ALL_MODELS = ["ridge","lasso","elastic_net","bayesian_ridge","ard",
              "svr","kernel_ridge","knn",
              "random_forest","extra_trees","hist_gradient_boosting","xgboost","lightgbm","mlp"]
MODELS_TO_RUN = ["ridge","lasso","elastic_net","bayesian_ridge","ard","svr","kernel_ridge","knn","mlp"]

PREV_BENCHMARK = None

RUN_PHC_ABLATION = False
MAKE_SUBMISSION = False
print("config ok | models:", MODELS_TO_RUN)


## 2. Load precomputed features


In [ ]:
PRECOMPUTED_DIR = CLONE_DIR / "precomputed"
train_csv = PRECOMPUTED_DIR / f"features_train_L{LIMIT}_T{TIME_BINS}.csv"
test_csv = PRECOMPUTED_DIR / f"features_test_T{TIME_BINS}.csv"

def _resolve(path):
    for cand in [path, OUTPUT_DIR / path.name, Path("/kaggle/input/ariel-features") / path.name]:
        if Path(cand).exists():
            return Path(cand)
    return path

train_csv, test_csv = _resolve(train_csv), _resolve(test_csv)
print("Train features:", train_csv, "| test:", test_csv, "(", test_csv.exists(), ")")


In [ ]:
features = pd.read_csv(train_csv)
targets = pd.read_csv(DATA_ROOT / "train.csv")
X, y, groups, target_columns = align_features_and_targets(features, targets)
Xv = X.to_numpy(dtype=float)
print("X:", Xv.shape, "| y:", y.shape, "| planets:", len(np.unique(groups)),
      "| features:", Xv.shape[1], "| targets:", y.shape[1])


## 2b. Diagnose if mean is learnable?


In [ ]:
from training import train_model

def per_wavelength_r2(name, n_components=N_COMPONENTS):
    res = train_model(Xv, y, model_name=name,
                      model_config=ModelConfig(n_components=n_components, random_state=RANDOM_STATE),
                      validation_fraction=0.25, groups=groups, random_state=RANDOM_STATE)
    yv = y[res.validation_index]
    pv = res.prediction.mu
    ss_res = ((yv - pv) ** 2).sum(axis=0)
    ss_tot = ((yv - yv.mean(axis=0)) ** 2).sum(axis=0)
    return 1.0 - ss_res / np.maximum(ss_tot, 1e-12)

r2s = {}
for name in ["ridge", "extra_trees"]:
    r2 = per_wavelength_r2(name)
    r2s[name] = r2
    print(f"{name:12s} mean R^2={r2.mean():+.3f} | median={np.median(r2):+.3f} | "
          f"% wavelength R^2>0: {(r2 > 0).mean() * 100:.0f}%")

plt.figure(figsize=(9, 3))
for name, r2 in r2s.items():
    plt.plot(r2, label=name, lw=1)
plt.axhline(0, color="k", lw=0.6)
plt.ylim(-1, 1)
plt.xlabel("wavelength index")
plt.ylabel("R^2 (val)")
plt.legend()
plt.title("Mean predictiveness per wavelength")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "diag_r2_per_wavelength.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Compare model families 


In [ ]:
Xb, yb, gb = Xv, y, groups
if MAX_PLANETS_BENCH is not None:
    keep = np.isin(groups, np.unique(groups)[:MAX_PLANETS_BENCH])
    Xb, yb, gb = Xv[keep], y[keep], groups[keep]
    print("Benchmark subset:", Xb.shape)

bench_cfg = ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE,
                        sigma_per_target=True, use_gpu=USE_GPU)
result = benchmark_models(Xb, yb, model_names=MODELS_TO_RUN, model_config=bench_cfg,
                          n_splits=N_SPLITS, groups=gb, random_state=RANDOM_STATE,
                          sigma_cal_fraction=SIGMA_CAL_FRACTION)
new_rows = result.to_frame()

RESULTS_CSV = OUTPUT_DIR / "benchmark.csv"
prev = None
for cand in [PREV_BENCHMARK, RESULTS_CSV]:
    if cand is not None and Path(cand).exists():
        prev = pd.read_csv(cand)
        break
table = (pd.concat([prev[~prev["model"].isin(new_rows["model"])], new_rows], ignore_index=True)
         if prev is not None else new_rows)
table = table.sort_values(["family", "ariel_gll_score"], ascending=[True, False]).reset_index(drop=True)
table.to_csv(RESULTS_CSV, index=False)

with pd.option_context("display.max_rows", None, "display.width", 200):
    print(table.to_string(index=False))
ok = table[table["status"] == "ok"]
if len(ok):
    b = ok.loc[ok["ariel_gll_score"].idxmax()]
    print("Best so far:", b["model"], "=", round(b["ariel_gll_score"], 4))
table


In [ ]:
ok = table[table["status"] == "ok"].sort_values("ariel_gll_score")
plt.figure(figsize=(8, max(3, 0.4 * len(ok))))
plt.barh(ok["model"], ok["ariel_gll_score"], color="steelblue")
plt.xlabel("Ariel GLL score (higher = better)")
plt.title("Model family comparison")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "benchmark_gll.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
from training import search_n_components

SEARCH_MODELS = ["bayesian_ridge", "ridge"]
N_COMPONENTS_GRID = [8, 12, 16, 24, 32, 48]

candidates = []
for name in SEARCH_MODELS:
    r = search_n_components(
        Xv, y, model_name=name, n_components_grid=N_COMPONENTS_GRID,
        base_config=ModelConfig(random_state=RANDOM_STATE, sigma_per_target=True, use_gpu=USE_GPU),
        n_splits=3, groups=groups, random_state=RANDOM_STATE,
        selection_metric="ariel_gll_score", sigma_cal_fraction=SIGMA_CAL_FRACTION,
    )
    candidates += r.candidates
bestc = max(candidates, key=lambda c: c.mean_metrics["ariel_gll_score"])
print("Best (n_components sweep):", bestc.model_name,
      "| n_components =", bestc.model_config.n_components,
      "| GLL =", round(bestc.mean_metrics["ariel_gll_score"], 4))


In [ ]:
if RUN_PHC_ABLATION:
    PHC_MODELS = ["bayesian_ridge", "extra_trees"]
    phc_modes = {
        "scalar":              ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, use_gpu=USE_GPU),
        "per_wavelength":      ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, use_gpu=USE_GPU, sigma_per_target=True),
        "feature_conditioned": ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, use_gpu=USE_GPU, sigma_feature_conditioned=True),
    }
    rows = []
    for mname in PHC_MODELS:
        for mode, cfg in phc_modes.items():
            m = cross_validate_model(Xv, y, model_name=mname, model_config=cfg,
                                     n_splits=3, groups=groups, random_state=RANDOM_STATE,
                                     sigma_cal_fraction=SIGMA_CAL_FRACTION).mean_metrics
            rows.append({"model": mname, "calibration": mode, "ariel_gll_score": m["ariel_gll_score"],
                         "gaussian_nll": m["gaussian_nll"], "coverage_1sigma": m["coverage_1sigma"]})
    phc_table = pd.DataFrame(rows)
    phc_table.to_csv(OUTPUT_DIR / "phc_ablation.csv", index=False)
    print(phc_table.to_string(index=False))
else:
    phc_table = None
    print("RUN_PHC_ABLATION=False -> bỏ qua ablation PHC")


In [ ]:
if RUN_PHC_ABLATION and phc_table is not None:
    pivot = phc_table.pivot(index="model", columns="calibration", values="ariel_gll_score")
    pivot = pivot[["scalar", "per_wavelength", "feature_conditioned"]]
    pivot.plot(kind="bar", figsize=(8, 4))
    plt.ylabel("Ariel GLL score")
    plt.title("Ablation PHC — sigma calibration")
    plt.xticks(rotation=0)
    plt.legend(title="calibration")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "phc_ablation_gll.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(pivot)
else:
    print("PHC ablation table is not available.")


In [ ]:
import joblib, gc

save_cfg = ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE,
                       sigma_per_target=True, use_gpu=USE_GPU)
VAL_FRACTION = 0.2

saved = []
for name in MODELS_TO_RUN:
    path = WEIGHTS_DIR / f"{name}.joblib"
    if path.exists():
        continue
    cfg = bestc.model_config if name == bestc.model_name else save_cfg
    try:
        res = train_model(Xv, y, model_name=name, model_config=cfg,
                          validation_fraction=VAL_FRACTION, groups=groups, random_state=RANDOM_STATE)
        joblib.dump({"model": res.model, "feature_columns": list(X.columns),
                     "target_columns": target_columns, "model_name": name,
                     "model_config": cfg, "val_metrics": res.evaluation.as_dict()}, path)
        saved.append(name)
        print(f"saved: {path}  (val GLL={res.evaluation.ariel_gll_score:.4f})")
    except Exception as exc:
        print("skip", name, "->", type(exc).__name__, exc)
    finally:
        gc.collect()

if MAKE_SUBMISSION:
    ens = build_gll_weighted_ensemble(
        Xv, y, model_names=("bayesian_ridge", "random_forest", "kernel_ridge"),
        model_config=ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, sigma_per_target=True, use_gpu=USE_GPU),
        validation_fraction=VAL_FRACTION, groups=groups, random_state=RANDOM_STATE)
    joblib.dump({"ensemble": ens.ensemble, "model_names": ens.model_names, "weights": ens.weights,
                 "feature_columns": list(X.columns), "target_columns": target_columns},
                WEIGHTS_DIR / "gll_weighted_ensemble.joblib")
    print("ensemble weights:", dict(zip(ens.model_names, np.round(ens.weights, 3))))
